![image_1780583484166.png](./image_1780583484166.png "image_1780583484166.png")

![image_1780583497964.png](./image_1780583497964.png "image_1780583497964.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

# Initialize Spark
spark = SparkSession.builder.appName("GrandSlams").getOrCreate()

# === Players DataFrame ===
players_data = [
    (1, "Novak Djokovic"),
    (2, "Rafael Nadal"),
    (3, "Roger Federer")
]

players_df = spark.createDataFrame(players_data, ["player_id", "player_name"])

# === Championships DataFrame ===
championships_data = [
    (2018, 1, 2, 1, 3),
    (2019, 1, 2, 2, 1),
    (2020, 1, 2, 3, 1)
]

championships_df = spark.createDataFrame(
    championships_data,
    ["year", "wimbledon", "fr_open", "us_open", "au_open"]
)

# Show the DataFrames
players_df.show()
championships_df.show()


In [0]:
all_winners_df = championships_df.select(f.col("wimbledon").alias("player_id")).union(
    (
        championships_df.select(f.col("fr_open").alias("player_id")).union(
            (
                championships_df.select(f.col("us_open").alias("player_id")).union(
                    championships_df.select(f.col("au_open").alias("player_id"))
                )
            )
        )
    )
)
result_df = (
    all_winners_df.join(players_df, on="player_id", how="inner")
    .groupBy("player_id", "player_name")
    .agg(f.count("*").alias("grand_slams_count"))
    .select("player_id", "player_name", "grand_slams_count")
    .orderBy(f.desc("grand_slams_count"), f.asc("player_name"))
)
display(result_df)